In [1]:
import json
import random
from pathlib import Path
from collections import defaultdict

CLEANED_PATH = Path("../data/processed/eval/eval_dataset_v6.json")
SAMPLE_OUTPUT_PATH = Path("../data/processed/eval/eval_dataset_sample_20.json")

SAMPLE_SIZE = 20    # 평가 데이터셋 분할 단위
RANDOM_SEED = 42


def infer_question_type(qid: str) -> str:
    qid = str(qid)

    if "fact_budget" in qid:
        return "fact_budget"
    if "llm_1" in qid:
        return "llm_1"
    if "llm_2" in qid:
        return "llm_2"

    return "unknown"


def sample_balanced_eval_dataset(
    input_path: Path,
    output_path: Path,
    sample_size: int = 30,
    random_seed: int = 42
):
    random.seed(random_seed)

    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    grouped = defaultdict(list)

    for item in data:
        question_type = infer_question_type(item.get("qid", ""))
        item["question_type"] = question_type
        grouped[question_type].append(item)

    target_types = ["fact_budget", "fact_deadline", "llm_1", "llm_2"]

    # 유형별 기본 할당량
    base_quota = sample_size // len(target_types)
    remainder = sample_size % len(target_types)

    sampled = []

    for i, question_type in enumerate(target_types):
        candidates = grouped.get(question_type, [])

        if not candidates:
            continue

        quota = base_quota + (1 if i < remainder else 0)
        quota = min(quota, len(candidates))

        sampled.extend(random.sample(candidates, quota))

    # 부족하면 전체에서 추가 샘플링
    if len(sampled) < sample_size:
        sampled_qids = {item["qid"] for item in sampled}
        remaining = [
            item for item in data
            if item["qid"] not in sampled_qids
        ]

        additional_count = min(sample_size - len(sampled), len(remaining))
        sampled.extend(random.sample(remaining, additional_count))

    # qid 기준 정렬하면 실험 결과 비교가 쉬움
    sampled = sorted(sampled, key=lambda x: x["qid"])

    output_path.parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(sampled, f, ensure_ascii=False, indent=2)

    print("샘플 평가셋 생성 완료")
    print(f"- 전체 정리 문항 수: {len(data)}")
    print(f"- 샘플 문항 수: {len(sampled)}")
    print(f"- 저장 파일: {output_path}")

    print("\n샘플 유형별 개수")
    type_counts = defaultdict(int)
    for item in sampled:
        type_counts[item["question_type"]] += 1

    for question_type, count in type_counts.items():
        print(f"- {question_type}: {count}")

    return sampled


sampled_data = sample_balanced_eval_dataset(
    input_path=CLEANED_PATH,
    output_path=SAMPLE_OUTPUT_PATH,
    sample_size=SAMPLE_SIZE,
    random_seed=RANDOM_SEED
)

샘플 평가셋 생성 완료
- 전체 정리 문항 수: 243
- 샘플 문항 수: 20
- 저장 파일: ../data/processed/eval/eval_dataset_sample_20.json

샘플 유형별 개수
- llm_1: 8
- fact_budget: 6
- llm_2: 6
